In [1]:
import pandas as pd
import numpy as np

# === Load the datasets
real_data = pd.read_csv("compas_test_with_model_predictions.csv")
synthetic_data = pd.read_csv("Our_Data_Compas_with_predictions.csv")

# === Define parameters
sensitive_col = 'race_African-American'
privileged_value = 1
model_cols = [
    'pred_decision_tree',
    'pred_logistic_regression',
    'pred_random_forest',
    'pred_svm',
    'pred_xgboost'
]

# === Fairness Metric Function
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluation function
def evaluate_fairness_for_all(data, true_label_col, source_name):
    results = []
    for model in model_cols:
        metrics = compute_fairness(
            y_true=data[true_label_col],
            y_pred=data[model],
            sensitive_attr=data[sensitive_col],
            privileged_value=privileged_value
        )
        results.append({"Model": model, "Source": source_name, **metrics})
    return results

# === Compute results
real_results = evaluate_fairness_for_all(real_data, 'true_label', 'Real')
synthetic_results = evaluate_fairness_for_all(synthetic_data, 'two_year_recid', 'Synthetic')

all_results = pd.DataFrame(real_results + synthetic_results)
pd.set_option("display.float_format", "{:.4f}".format)

# === Print and save
print(all_results.to_string(index=False))
all_results.to_csv("fairness_results_compas.csv", index=False)
print("\n✅ Results saved to: fairness_results_compas.csv")


c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


                   Model    Source    DPD     DI  ∆TPR (EoO)    ∆FPR    ∆PPV  ∆Accuracy    EOD
      pred_decision_tree      Real 0.1148 0.7401      0.1053  0.0844  0.0821    -0.0197 0.1053
pred_logistic_regression      Real 0.2664 0.4526      0.3017  0.1769  0.0216     0.0143 0.3017
      pred_random_forest      Real 0.1429 0.7189      0.1221  0.1147  0.0671    -0.0171 0.1221
                pred_svm      Real 0.2633 0.4556      0.3072  0.1732  0.0507     0.0181 0.3072
            pred_xgboost      Real 0.2018 0.5690      0.2590  0.1057  0.1207     0.0354 0.2590
      pred_decision_tree Synthetic 0.0483 1.1140      0.0741 -0.0238 -0.0106     0.0564 0.0741
pred_logistic_regression Synthetic 0.0407 1.0984      0.1586 -0.0815  0.0698     0.1292 0.1586
      pred_random_forest Synthetic 0.0668 1.1576      0.0302 -0.0031 -0.0271     0.0225 0.0302
                pred_svm Synthetic 0.0487 1.1358      0.1026 -0.0615  0.0530     0.1027 0.1026
            pred_xgboost Synthetic 0.0483 1.1140  

In [5]:
import pandas as pd
import numpy as np

# === Load the datasets
real_data = pd.read_csv("compas_test_with_model_predictions.csv")
synthetic_data = pd.read_csv("Synt_Data_DECAF_ with_predictions.csv")

# === Define parameters
sensitive_col = 'race_African-American'
privileged_value = 1
model_cols = [
    'pred_decision_tree',
    'pred_logistic_regression',
    'pred_random_forest',
    'pred_svm',
    'pred_xgboost'
]

# === Fairness Metric Function
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluation function
def evaluate_fairness_for_all(data, true_label_col, source_name):
    results = []
    for model in model_cols:
        metrics = compute_fairness(
            y_true=data[true_label_col],
            y_pred=data[model],
            sensitive_attr=data[sensitive_col],
            privileged_value=privileged_value
        )
        results.append({"Model": model, "Source": source_name, **metrics})
    return results

# === Compute results
real_results = evaluate_fairness_for_all(real_data, 'true_label', 'Real')
synthetic_results = evaluate_fairness_for_all(synthetic_data, 'two_year_recid', 'Synthetic')



all_results = pd.DataFrame(real_results + synthetic_results)
pd.set_option("display.float_format", "{:.4f}".format)

# === Print and save
print(all_results.to_string(index=False))
all_results.to_csv("fairness_results_compasdecaf.csv", index=False)
print("\n✅ Results saved to: fairness_results_compas.csv")


                   Model    Source    DPD     DI  ∆TPR (EoO)    ∆FPR    ∆PPV  ∆Accuracy    EOD
      pred_decision_tree      Real 0.1148 0.7401      0.1053  0.0844  0.0821    -0.0197 0.1053
pred_logistic_regression      Real 0.2664 0.4526      0.3017  0.1769  0.0216     0.0143 0.3017
      pred_random_forest      Real 0.1429 0.7189      0.1221  0.1147  0.0671    -0.0171 0.1221
                pred_svm      Real 0.2633 0.4556      0.3072  0.1732  0.0507     0.0181 0.3072
            pred_xgboost      Real 0.2018 0.5690      0.2590  0.1057  0.1207     0.0354 0.2590
      pred_decision_tree Synthetic 0.0636 1.1296      0.0102 -0.1598  0.0729     0.0741 0.1598
pred_logistic_regression Synthetic 0.1051 1.2522     -0.0603 -0.1593  0.0653     0.0350 0.1593
      pred_random_forest Synthetic 0.0213 0.9622      0.0946 -0.0747  0.0630     0.0849 0.0946
                pred_svm Synthetic 0.0266 1.0957     -0.0848  0.0546 -0.1167    -0.0668 0.0848
            pred_xgboost Synthetic 0.0081 1.0147  

In [6]:
import pandas as pd
import numpy as np

# === Load the datasets
real_data = pd.read_csv("compas_test_with_model_predictions.csv")
synthetic_data = pd.read_csv("CLLM_with_predictions_COMPAS.csv")

# === Define parameters
sensitive_col = 'race_African-American'
privileged_value = 1
model_cols = [
    'pred_decision_tree',
    'pred_logistic_regression',
    'pred_random_forest',
    'pred_svm',
    'pred_xgboost'
]

# === Fairness Metric Function
def compute_fairness(y_true, y_pred, sensitive_attr, privileged_value):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    sensitive_attr = np.array(sensitive_attr)

    mask_priv = sensitive_attr == privileged_value
    mask_unpriv = ~mask_priv

    p_priv = y_pred[mask_priv].mean()
    p_unpriv = y_pred[mask_unpriv].mean()
    dpd = abs(p_priv - p_unpriv)
    di = p_unpriv / p_priv if p_priv > 0 else float('inf')

    def conf_values(y_true, y_pred):
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        return TP, TN, FP, FN

    TP_p, TN_p, FP_p, FN_p = conf_values(y_true[mask_priv], y_pred[mask_priv])
    TP_u, TN_u, FP_u, FN_u = conf_values(y_true[mask_unpriv], y_pred[mask_unpriv])

    tpr_p = TP_p / (TP_p + FN_p) if TP_p + FN_p > 0 else 0
    tpr_u = TP_u / (TP_u + FN_u) if TP_u + FN_u > 0 else 0
    delta_tpr = tpr_p - tpr_u

    fpr_p = FP_p / (FP_p + TN_p) if FP_p + TN_p > 0 else 0
    fpr_u = FP_u / (FP_u + TN_u) if FP_u + TN_u > 0 else 0
    delta_fpr = fpr_p - fpr_u

    ppv_p = TP_p / (TP_p + FP_p) if TP_p + FP_p > 0 else 0
    ppv_u = TP_u / (TP_u + FP_u) if TP_u + FP_u > 0 else 0
    delta_ppv = ppv_p - ppv_u

    acc_p = (TP_p + TN_p) / (TP_p + TN_p + FP_p + FN_p) if TP_p + TN_p + FP_p + FN_p > 0 else 0
    acc_u = (TP_u + TN_u) / (TP_u + TN_u + FP_u + FN_u) if TP_u + TN_u + FP_u + FN_u > 0 else 0
    delta_acc = acc_p - acc_u

    eod = max(abs(delta_tpr), abs(delta_fpr))
    eoo = delta_tpr

    return {
        "DPD": dpd,
        "DI": di,
        "∆TPR (EoO)": eoo,
        "∆FPR": delta_fpr,
        "∆PPV": delta_ppv,
        "∆Accuracy": delta_acc,
        "EOD": eod
    }

# === Evaluation function
def evaluate_fairness_for_all(data, true_label_col, source_name):
    results = []
    for model in model_cols:
        metrics = compute_fairness(
            y_true=data[true_label_col],
            y_pred=data[model],
            sensitive_attr=data[sensitive_col],
            privileged_value=privileged_value
        )
        results.append({"Model": model, "Source": source_name, **metrics})
    return results

# === Compute results
real_results = evaluate_fairness_for_all(real_data, 'true_label', 'Real')
synthetic_results = evaluate_fairness_for_all(synthetic_data, 'true_label', 'Synthetic')

all_results = pd.DataFrame(real_results + synthetic_results)
pd.set_option("display.float_format", "{:.4f}".format)

# === Print and save
print(all_results.to_string(index=False))
all_results.to_csv("fairness_results_compas_cllm.csv", index=False)
print("\n✅ Results saved to: fairness_results_compas.csv")


                   Model    Source    DPD     DI  ∆TPR (EoO)    ∆FPR   ∆PPV  ∆Accuracy    EOD
      pred_decision_tree      Real 0.1148 0.7401      0.1053  0.0844 0.0821    -0.0197 0.1053
pred_logistic_regression      Real 0.2664 0.4526      0.3017  0.1769 0.0216     0.0143 0.3017
      pred_random_forest      Real 0.1429 0.7189      0.1221  0.1147 0.0671    -0.0171 0.1221
                pred_svm      Real 0.2633 0.4556      0.3072  0.1732 0.0507     0.0181 0.3072
            pred_xgboost      Real 0.2018 0.5690      0.2590  0.1057 0.1207     0.0354 0.2590
      pred_decision_tree Synthetic 0.0751 1.1317     -0.1038 -0.1819 0.1347     0.0160 0.1819
pred_logistic_regression Synthetic 0.0203 0.9650     -0.0082 -0.1167 0.1024     0.0384 0.1167
      pred_random_forest Synthetic 0.0016 0.9971     -0.0269 -0.1114 0.1064     0.0253 0.1114
                pred_svm Synthetic 0.0081 0.9862     -0.0374 -0.1193 0.0952     0.0234 0.1193
            pred_xgboost Synthetic 0.0189 0.9679     -0.0418